# ⚡ High-Speed Manifest Consolidator (Fast Manifest-Based Scan)
### Consolidates `train`, `val`, and `test` manifests in ~5 SECONDS using Shard Manifests

Sa halip na mabagal na `glob` sa Google Drive, binabasa nito ang mga nagawang **`shard_XXXX_manifest.csv`** at diretsong binubuo ang 3 official final manifests:
- `final_train_manifest.csv`
- `final_val_manifest.csv`
- `final_test_manifest.csv`

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## Step 2: Clone Baseline Repository

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
print('✅ Repository ready!')

## Step 3: Fast Manifest Consolidation (~5 Seconds)

In [ ]:
import os, csv, glob
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/Baseline preprocessed')
OUTPUT_CSV_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/manifests')
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Preprocessed Root : {DRIVE_ROOT}")
print(f"📁 Output Manifests  : {OUTPUT_CSV_DIR}")
print("=" * 85)

SPLIT_CONFIGS = {
    'train': {'folder': DRIVE_ROOT / 'TRAIN', 'target': 14815},
    'val':   {'folder': DRIVE_ROOT / 'VAL',   'target': 1457},
    'test':  {'folder': DRIVE_ROOT / 'TEST',  'target': 1469}
}

summary_stats = []

for split_name, cfg in SPLIT_CONFIGS.items():
    split_dir = cfg['folder']
    target_count = cfg['target']
    
    print(f"\n🔍 Processing [{split_name.upper()}] from {split_dir}...")
    
    # Find all shard manifest CSVs generated by run_shard.py
    manifest_files = []
    if split_dir.exists():
        for root, dirs, files in os.walk(str(split_dir)):
            for f in files:
                if f.endswith('_manifest.csv'):
                    manifest_files.append(os.path.join(root, f))
                    
    print(f"   Found {len(manifest_files)} shard manifest CSVs")
    
    all_rows = []
    fieldnames = []
    seen_cids = set()
    
    for mf in sorted(manifest_files):
        try:
            with open(mf, newline='', encoding='utf-8') as f_in:
                reader = csv.DictReader(f_in)
                if not fieldnames:
                    fieldnames = reader.fieldnames
                for r in reader:
                    cid = r.get('clip_id')
                    if cid and cid not in seen_cids:
                        seen_cids.add(cid)
                        all_rows.append(r)
        except Exception as e:
            print(f"   [Warning] Error reading {mf}: {e}")
            
    # Save consolidated split CSV
    out_csv_path = OUTPUT_CSV_DIR / f'final_{split_name}_manifest.csv'
    if all_rows and fieldnames:
        with open(out_csv_path, 'w', newline='', encoding='utf-8') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)
            
    pct = (len(all_rows) / target_count) * 100 if target_count > 0 else 0
    summary_stats.append({
        'Split': split_name.upper(),
        'Target Clips': target_count,
        'Processed Clips': len(all_rows),
        'Completion (%)': f"{pct:.2f}%",
        'Output CSV File': out_csv_path.name
    })

print("\n" + "=" * 85)
print("                      🏆 FINAL CONSOLIDATED MANIFEST REPORT 🏆")
print("=" * 85)
df_res = pd.DataFrame(summary_stats)
print(df_res.to_string(index=False))
print("=" * 85)
print(f"\n📍 Successfully generated all 3 final CSVs in:")
print(f"   -> {OUTPUT_CSV_DIR}")
print("=" * 85)